In [1]:
from keras.datasets import mnist
from keras.utils import to_categorical

In [2]:
#Showing directories’ size
import os, shutil

train_dir ='./images/train'
validation_dir = './images/val'
test_dir = './images/test'

train_corona_virus_d = './images/train/Corona_Virus_Disease'
train_normal = './images/train/Normal'
train_tuberculosis= './images/train/Tuberculosis'

val_corona_virus_d = './images/val/Corona_Virus_Disease'
val_normal = './images/val/Normal'
val_tuberculosis= './images/val/Tuberculosis'

test_corona_virus_d = './images/test/Corona_Virus_Disease'
test_normal = './images/test/Normal'
test_tuberculosis= './images/test/Tuberculosis'


print('total train corona virus images:', len(os.listdir(train_corona_virus_d)))
print('total train normal images:', len(os.listdir(train_normal)))
print('total train tuberculosis images:', len(os.listdir(train_tuberculosis)))


print('total validation corona virus images:', len(os.listdir(val_corona_virus_d)))
print('total validation normal images:', len(os.listdir(val_normal)))
print('total validation tuberculosis images:', len(os.listdir(val_tuberculosis)))


print('total testing organic corona virus img:', len(os.listdir(test_corona_virus_d)))
print('total testing recicle normal imgs:', len(os.listdir(test_normal)))
print('total testing recicle tuberculosis img:', len(os.listdir(test_tuberculosis)))

total train corona virus images: 1218
total train normal images: 1207
total train tuberculosis images: 1220
total validation corona virus images: 406
total validation normal images: 402
total validation tuberculosis images: 406
total testing organic corona virus img: 407
total testing recicle normal imgs: 404
total testing recicle tuberculosis img: 408


In [3]:
from keras.utils import image_dataset_from_directory

IMG_SIZE = 150
train_dataset = image_dataset_from_directory(
    train_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    label_mode='categorical',
    seed = 100
)

validation_dataset = image_dataset_from_directory(
    validation_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=64,
    label_mode='categorical',
    seed = 100
)

test_dataset = image_dataset_from_directory(
    test_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=64,
    label_mode='categorical',
    seed = 100
)

Found 3645 files belonging to 3 classes.
Found 1214 files belonging to 3 classes.
Found 1219 files belonging to 3 classes.


In [4]:
from keras.applications.vgg16 import VGG16

# Crear el modelo base de VGG16 preentrenado en ImageNet
conv_base = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

In [5]:
# Congelar las capas de la base convolucional para que no se entrenen
conv_base.trainable = False

In [6]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
import numpy as np

def objective(trial):
    opt_num_hidden_dense_units = trial.suggest_int("opt_num_hidden_dense_units", 10, 100)
    opt_lr = trial.suggest_float("opt_lr", 1e-6, 1e-2, log=True)
    opt_bs = trial.suggest_int("opt_bs", 16, 128)
    data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.2),
    ]
    )
    # Construir el modelo
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = conv_base(x, training=False)
    x = layers.Flatten()(x)
    x = layers.Dense(opt_num_hidden_dense_units, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(3, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    # Compilar el modelo
    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=opt_lr),
        metrics=['acc'])
    history = model.fit(
        train_dataset,
        epochs=30,
        validation_data=validation_dataset,
        verbose=1,
        batch_size=opt_bs)
    min_val_loss = np.amin(history.history["val_loss"])
    return min_val_loss
    



In [7]:
import optuna as opt
study = opt.create_study()
study.optimize(objective, n_trials=5)


C:\Users\alfre\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2024-07-05 01:34:21,514] A new study created in memory with name: no-name-5d848936-56fd-4798-84f3-985ca25ef0e2


Epoch 1/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 123s 1s/step - acc: 0.6285 - loss: 8.1962 - val_acc: 0.7504 - val_loss: 0.5001
Epoch 2/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.7040 - loss: 0.6997 - val_acc: 0.8278 - val_loss: 0.4838
Epoch 3/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.6887 - loss: 0.7341 - val_acc: 0.8748 - val_loss: 0.4120
Epoch 4/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.7190 - loss: 0.6988 - val_acc: 0.8649 - val_loss: 0.3863
Epoch 5/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.7809 - loss: 0.5387 - val_acc: 0.8707 - val_loss: 0.3641
Epoch 6/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - acc: 0.7644 - loss: 0.5650 - val_acc: 0.9250 - val_loss: 0.2796
Epoch 7/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 133s 1s/step - acc: 0.8139 - loss: 0.4868 - val_acc: 0.9481 - val_loss: 0.2288
Epoch 8/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 125s 1s/step - acc: 0.8510 - loss: 0.4250 - val_acc: 0.9423 - val_loss: 0.2301
Epoch 9/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/

[I 2024-07-05 02:35:17,386] Trial 0 finished with value: 0.13038527965545654 and parameters: {'opt_num_hidden_dense_units': 46, 'opt_lr': 0.0032175167961439335, 'opt_bs': 23}. Best is trial 0 with value: 0.13038527965545654.


Epoch 1/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.3410 - loss: 11.9693 - val_acc: 0.4094 - val_loss: 5.5496
Epoch 2/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.3733 - loss: 8.8866 - val_acc: 0.5231 - val_loss: 4.0020
Epoch 3/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.4050 - loss: 7.3322 - val_acc: 0.5799 - val_loss: 3.2574
Epoch 4/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 123s 1s/step - acc: 0.4580 - loss: 5.6414 - val_acc: 0.6219 - val_loss: 2.7548
Epoch 5/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - acc: 0.4978 - loss: 4.8172 - val_acc: 0.6532 - val_loss: 2.3943
Epoch 6/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.5078 - loss: 4.2308 - val_acc: 0.6746 - val_loss: 2.1010
Epoch 7/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.5303 - loss: 3.6962 - val_acc: 0.6878 - val_loss: 1.8961
Epoch 8/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.5486 - loss: 3.2538 - val_acc: 0.7059 - val_loss: 1.7157
Epoch 9/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s

[I 2024-07-05 03:34:52,078] Trial 1 finished with value: 0.5940389037132263 and parameters: {'opt_num_hidden_dense_units': 30, 'opt_lr': 3.153246626613824e-06, 'opt_bs': 127}. Best is trial 0 with value: 0.13038527965545654.


...

[I 2024-07-05 04:38:33,591] Trial 2 finished with value: 0.2230062484741211 and parameters: {'opt_num_hidden_dense_units': 47, 'opt_lr': 1.4285791144943133e-05, 'opt_bs': 114}. Best is trial 0 with value: 0.13038527965545654.


Epoch 1/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 167s 1s/step - acc: 0.3520 - loss: 13.8176 - val_acc: 0.3666 - val_loss: 7.0408
Epoch 2/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 150s 1s/step - acc: 0.4640 - loss: 7.7794 - val_acc: 0.4811 - val_loss: 4.6157
Epoch 3/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 128s 1s/step - acc: 0.5051 - loss: 6.4406 - val_acc: 0.5774 - val_loss: 3.4609
Epoch 4/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 127s 1s/step - acc: 0.5608 - loss: 5.2009 - val_acc: 0.6590 - val_loss: 2.6971
Epoch 5/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 126s 1s/step - acc: 0.5812 - loss: 4.7607 - val_acc: 0.6960 - val_loss: 2.3051
Epoch 6/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 126s 1s/step - acc: 0.6200 - loss: 4.0358 - val_acc: 0.7339 - val_loss: 2.0028
Epoch 7/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 127s 1s/step - acc: 0.6250 - loss: 3.7913 - val_acc: 0.7611 - val_loss: 1.7901
Epoch 8/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 128s 1s/step - acc: 0.6479 - loss: 3.6123 - val_acc: 0.7751 - val_loss: 1.6367
Epoch 9/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 127s 1s

[I 2024-07-05 05:42:45,036] Trial 3 finished with value: 0.5906383395195007 and parameters: {'opt_num_hidden_dense_units': 98, 'opt_lr': 3.1146681196005003e-06, 'opt_bs': 32}. Best is trial 0 with value: 0.13038527965545654.


Epoch 1/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.4353 - loss: 3.0966 - val_acc: 0.8830 - val_loss: 0.5509
Epoch 2/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.6648 - loss: 0.7521 - val_acc: 0.8270 - val_loss: 0.5517
Epoch 3/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.6769 - loss: 0.7347 - val_acc: 0.8888 - val_loss: 0.4579
Epoch 4/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - acc: 0.6801 - loss: 0.6971 - val_acc: 0.8583 - val_loss: 0.4525
Epoch 5/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - acc: 0.7116 - loss: 0.6550 - val_acc: 0.9044 - val_loss: 0.3961
Epoch 6/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.7038 - loss: 0.6516 - val_acc: 0.9349 - val_loss: 0.3501
Epoch 7/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - acc: 0.6838 - loss: 0.6919 - val_acc: 0.9234 - val_loss: 0.3568
Epoch 8/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - acc: 0.7012 - loss: 0.6632 - val_acc: 0.7479 - val_loss: 0.5662
Epoch 9/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/

[I 2024-07-05 06:42:17,910] Trial 4 finished with value: 0.30627843737602234 and parameters: {'opt_num_hidden_dense_units': 24, 'opt_lr': 0.003627921400063663, 'opt_bs': 57}. Best is trial 0 with value: 0.13038527965545654.
